<a href="https://colab.research.google.com/github/heewonLEE2/Paper-Implementations/blob/main/NLP/LSTM_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✅ LSTM을 이용한 IMDB 감성 분류 (Many-to-one)

## 개요
PyTorch의 `nn.LSTM`을 사용해 IMDB 영화 리뷰 데이터셋을 긍정/부정으로 분류하는 모델을 구현합니다.  

---

## 모델 구조
```
텍스트 입력 → Embedding → LSTM → Dropout → FC → 긍정/부정
```

| 구성 요소 | 설정값 |
|-----------|--------|
| Embedding dim | 128 |
| Hidden size | 256 |
| num_layers | 1 |
| Dropout | 0.5 |
| 출력 클래스 | 2 (긍정 / 부정) |

---

## 데이터 전처리
- **데이터셋** : HuggingFace `datasets`의 IMDB (train 25,000 / test 25,000)
- **토크나이저** : 소문자 변환 후 공백 기준 분리 (`text.lower().split()`)
- **Vocab 크기** : 상위 25,000개 단어 + `[PAD]`(0) + `[UNK]`(1) = 25,002
- **최대 시퀀스 길이** : 256 (초과 시 Truncation, 부족 시 PAD로 채움)

---

## 학습 설정
- **Loss** : `nn.CrossEntropyLoss`
- **Optimizer** : `Adam (lr=1e-3)`
- **Batch size** : 64
- **Epochs** : 5

---

## 핵심 학습 포인트
1. `nn.LSTM`의 3가지 반환값 (`output`, `h_n`, `c_n`) 의 차이
2. `batch_first=True` 옵션과 input shape
3. Many-to-one에서 `h_n[-1]` 을 FC에 넘기는 이유
4. `model.train()` / `model.eval()` 모드 차이
5. LSTM 학습 = 4개 게이트(F/I/O/Cell)의 가중치 W, b 최적화

# 📚 핵심 학습 포인트 정리

---

## 1. `nn.LSTM`의 3가지 반환값 차이
```python
output, (h_n, c_n) = nn.LSTM(input)
```

| 반환값 | shape | 의미 |
|--------|-------|------|
| `output` | `(batch, seq_len, hidden)` | **모든 타임스텝**의 hidden state → Many-to-many에서 사용 |
| `h_n` | `(num_layers, batch, hidden)` | **마지막 타임스텝**의 hidden state → Many-to-one에서 사용 |
| `c_n` | `(num_layers, batch, hidden)` | **마지막 타임스텝**의 cell state (장기 기억) |

---

## 2. `batch_first=True` 옵션과 input shape

PyTorch LSTM의 기본 input shape은 `(seq_len, batch, feature)` 로 불편하게 설계되어 있음.  
`batch_first=True` 설정 시 일반적인 데이터 처리 순서인 `(batch, seq_len, feature)` 로 변경됨.
```python
self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
```

> ⚠️ `batch_first=True` 를 설정해도 `h_n`, `c_n` 의 shape은 항상 `(num_layers, batch, hidden)` 으로 고정

---

## 3. Many-to-one에서 `h_n[-1]`을 FC에 넘기는 이유

Many-to-one은 시퀀스 전체를 읽고 **하나의 결과**만 출력하는 태스크.  
`h_n` 은 마지막 타임스텝의 hidden state를 레이어별로 저장하므로,  
`h_n[-1]` = **마지막 레이어의, 마지막 타임스텝** hidden state를 FC에 넘김.
```python
output, (h_n, c_n) = self.lstm(embedded)
last_hidden = h_n[-1]   # (batch, hidden) → FC 입력
```

> `h_n[-1]` 을 쓰면 `num_layers` 값에 무관하게 항상 마지막 레이어를 가져올 수 있어 안전함

---

## 4. `model.train()` / `model.eval()` 모드 차이

| | `model.train()` | `model.eval()` |
|--|----------------|----------------|
| **Dropout** | 랜덤하게 뉴런 비활성화 ✅ | 모든 뉴런 사용 |
| **사용 시점** | 학습 시 | 검증 / 추론 시 |
```python
# 반드시 모드 전환을 명시적으로 해줘야 함
model.train()   # 학습 루프 전
model.eval()    # 검증 / 추론 루프 전
```

> ⚠️ `eval()` 모드인 채로 학습하면 Dropout이 꺼져 과적합 발생 가능

---

## 5. LSTM 학습 = 4개 게이트의 가중치 W, b 최적화

LSTM 학습이란 역전파를 통해 아래 4개 구성요소의 가중치를 조정하는 것.

| 구성요소 | 수식 | 역할 |
|----------|------|------|
| Forget Gate | $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ | 과거 기억 중 버릴 것 결정 |
| Input Gate | $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ | 새 정보 중 저장할 것 결정 |
| Cell 후보값 | $\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$ | 실제 저장할 후보값 생성 |
| Output Gate | $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ | 출력할 정보 결정 |

각 가중치는 두 가지 입력에 대해 학습됨.
- `W_x` : 현재 입력 $x_t$ 에 대한 가중치
- `W_h` : 이전 hidden state $h_{t-1}$ 에 대한 가중치


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter

# -------------------------------------------------------
# 1. 데이터 로드
# -------------------------------------------------------
dataset = load_dataset("imdb")
train_data = dataset["train"]  # 25,000개
test_data  = dataset["test"]   # 25,000개

In [2]:
# -------------------------------------------------------
# 2. 토크나이저
# -------------------------------------------------------
tokenizer = lambda text: text.lower().split()

In [12]:
train_data

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [3]:
# -------------------------------------------------------
# 3. Vocabulary 구축
# -------------------------------------------------------
MAX_VOCAB = 25000
PAD_IDX   = 0
UNK_IDX   = 1

counter = Counter()
for item in train_data:
    counter.update(tokenizer(item["text"]))

# 상위 25,000개 단어만 선택
vocab = {"[PAD]": PAD_IDX, "[UNK]": UNK_IDX}
for word, _ in counter.most_common(MAX_VOCAB):
    vocab[word] = len(vocab)

In [ ]:
vocab
'''
단어 사전 구축
{'[PAD]': 0,
 '[UNK]': 1,
 'the': 2,
 'a': 3,
 ...
'''

In [4]:
# -------------------------------------------------------
# 4. 텍스트 → 정수 인코딩 + 패딩/트런케이션
# -------------------------------------------------------
MAX_LEN = 256

def encode(text):
    tokens = tokenizer(text)[:MAX_LEN]                          # 트런케이션
    ids    = [vocab.get(t, UNK_IDX) for t in tokens]           # 정수 인코딩
    ids   += [PAD_IDX] * (MAX_LEN - len(ids))                  # 패딩
    return ids

In [5]:
# -------------------------------------------------------
# 5. Dataset 클래스
# -------------------------------------------------------
class IMDBDataset(Dataset):
    def __init__(self, data):
        self.texts  = [torch.tensor(encode(item["text"])) for item in data]
        self.labels = [torch.tensor(item["label"]) for item in data]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

train_dataset = IMDBDataset(train_data)
test_dataset  = IMDBDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

In [15]:
train_dataset.__len__()

25000

In [16]:
train_dataset[0]

(tensor([    9,  1486,     9,   226,     1,    34,    57,   433,  1495,    77,
             5,    35,     2,  9840,    11,  3471,    12,    50,    12,    14,
            82,   702,     8,     1,     9,    81,   510,    11,    29,    82,
            12,    14,     1,    31,  2443, 14300,    46,    12,   125,   737,
             6,  2764,    10,  4218,  1935,    99,     3,   376,     5,   129,
          1127,     1,     9,    61,    62,     6,    67,    10,    16, 18115,
            13,    93,   131,     7,  6447,   197,     3,   185,  4380,   659,
          1669,   699,  5601,    36,   438,     6,   781,   292,    53,    64,
            43,   506,     8,   928,    53,   438,     6,  1182,    42, 15381,
             6,   242,    45,   391,     5,   797,    19,    48,     2,   952,
             1,   199,    43,   721,   954,  1600,   130,    15,     2,  3472,
           408,     4,  1977,  1600,     8,     2,  2351,  7328,     8,   187,
          2170,  9841,     4,  2268,     1,     5,  

In [6]:
# -------------------------------------------------------
# 6. 모델 정의
# -------------------------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes=2):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.lstm      = nn.LSTM(embedding_dim, hidden_size, num_layers=1, batch_first=True)
        self.dropout   = nn.Dropout(0.5)
        self.fc        = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded    = self.embedding(x)
        _, (h_n, _) = self.lstm(embedded)
        last_hidden = h_n[-1]
        dropped     = self.dropout(last_hidden)
        logits      = self.fc(dropped)
        return logits

In [7]:
# -------------------------------------------------------
# 7. Train / Test loop
# -------------------------------------------------------
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss    = 0
    total_correct = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss    += loss.item()
        preds          = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()

    return total_loss / len(dataloader), total_correct / len(dataloader.dataset)


def test(model, dataloader, criterion, device):
    model.eval()
    total_loss    = 0
    total_correct = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            logits = model(inputs)
            loss   = criterion(logits, labels)

            total_loss    += loss.item()
            preds          = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()

    return total_loss / len(dataloader), total_correct / len(dataloader.dataset)

In [9]:
# -------------------------------------------------------
# 8. 실행
# -------------------------------------------------------
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model     = LSTMClassifier(
                vocab_size    = len(vocab),     # 25,002
                embedding_dim = 128,
                hidden_size   = 256
            ).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 20):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss,  test_acc  = test(model,  test_loader,  criterion, device)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Epoch 01 | Train Loss: 0.6938, Train Acc: 0.5142 | Test Loss: 0.6935, Test Acc: 0.5088
Epoch 02 | Train Loss: 0.6954, Train Acc: 0.5297 | Test Loss: 0.6945, Test Acc: 0.5066
Epoch 03 | Train Loss: 0.6862, Train Acc: 0.5320 | Test Loss: 0.6956, Test Acc: 0.5108
Epoch 04 | Train Loss: 0.6650, Train Acc: 0.5796 | Test Loss: 0.6909, Test Acc: 0.5434
Epoch 05 | Train Loss: 0.6460, Train Acc: 0.6075 | Test Loss: 0.6890, Test Acc: 0.5919
Epoch 06 | Train Loss: 0.6281, Train Acc: 0.5970 | Test Loss: 0.6652, Test Acc: 0.6650
Epoch 07 | Train Loss: 0.6726, Train Acc: 0.5773 | Test Loss: 0.7081, Test Acc: 0.5153
Epoch 08 | Train Loss: 0.6469, Train Acc: 0.5707 | Test Loss: 0.7097, Test Acc: 0.5141
Epoch 09 | Train Loss: 0.6228, Train Acc: 0.5958 | Test Loss: 0.7363, Test Acc: 0.5183
Epoch 10 | Train Loss: 0.5974, Train Acc: 0.6035 | Test Loss: 0.7523, Test Acc: 0.5218
Epoch 11 | Train Loss: 0.5766, Train Acc: 0.6128 | Test Loss: 0.7715, Test Acc: 0.5282
Epoch 12 | Train Loss: 0.5170, Train Acc: 0

In [20]:
def predict(model, texts, vocab, tokenizer, device):
    model.eval()

    with torch.no_grad():
        for text in texts:
            ids    = encode(text)                               # ① 인코딩
            tensor = torch.tensor(ids).unsqueeze(0).to(device) # ② (1, seq_len)

            logits = model(tensor)                             # ③ 추론
            probs  = torch.softmax(logits, dim=1)             # ④ 확률 변환
            pred   = probs.argmax(dim=1).item()               # ⑤ 예측 클래스
            label  = "긍정 😊" if pred == 1 else "부정 😞"

            print(f"텍스트 : {text[:60]}...")
            print(f"예측   : {label}")
            print(f"확률   : 부정 {probs[0][0]:.4f} | 긍정 {probs[0][1]:.4f}")
            print("-" * 70)

# -------------------------------------------------------
# 테스트 문장
# -------------------------------------------------------
texts = [
    "This movie was absolutely fantastic! I loved every moment of it.",
    "This was the worst movie I have ever seen. Terrible acting and boring plot."
]

predict(model, texts, vocab, tokenizer, device)

텍스트 : This movie was absolutely fantastic! I loved every moment of...
예측   : 긍정 😊
확률   : 부정 0.0058 | 긍정 0.9942
----------------------------------------------------------------------
텍스트 : This was the worst movie I have ever seen. Terrible acting a...
예측   : 부정 😞
확률   : 부정 0.9955 | 긍정 0.0045
----------------------------------------------------------------------
